In [2]:
from dotenv import load_dotenv
from langchain_tavily import TavilySearch
load_dotenv()
search_tool=TavilySearch()
response=search_tool.invoke("梅西是谁")
print(response )

{'query': '梅西是谁', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://zh.wikipedia.org/zh-hans/%E5%88%A9%E6%98%82%E5%86%85%E5%B0%94%C2%B7%E6%A2%85%E8%A5%BF', 'title': '利昂内尔·梅西 - 维基百科，自由的百科全书', 'content': '| “Lionel Messi”的各地常用译名 |\n\n| 中国大陆 | 梅西 |\n| 台湾 | 梅西 |\n| 香港 | 美斯 |\n| 澳门 | 美斯 |\n\n利昂内尔·安德烈斯·“利奥”·梅西（西班牙语：Lionel Andrés "Leo" Messi，发音：(/wiki/Help:%E8%A5%BF%E7%8F%AD%E7%89%99%E8%AF%AD%E5%9B%BD%E9%99%85%E9%9F%B3%E6%A0%87 "Help:西班牙语国际音标") ⓘ；1987年6月24日—），阿根廷男子足球员，现效力于美国职业足球大联盟的迈阿密国际足球俱乐部并任队长，前阿根廷国家足球队队长，司职包括翼锋、前腰、伪九号#偽九號 "前锋 (足球)")等位置。其被视为足球史上最伟大的球员。\n\n梅西拥有加泰罗尼亚及意大利血统，他的职业生涯从阿根廷球会纽韦尔老男孩开始，在13岁时转会到了西甲的巴塞罗那足球俱乐部的青训营拉玛西亚，并于2005年至2021年间长年效力于巴塞罗那。2021年他转投法甲球队巴黎圣日耳曼，在巴黎渡过两个赛季后，他于2023年转投美足联球队迈阿密国际至今。 [...] 梅西是现代足球史上，少数能够获得个人与团体名誉大满贯的足球运动员。在个人名誉方面，目前他共获得8座金球奖 "金球奖 (足球)")、2座世界杯金球奖、8次世界足球先生、2次劳伦斯世界体育奖年度最佳男子运动员及6座欧洲金靴奖。而在团体赛事方面，他在俱乐部获得10次西班牙甲组足球联赛冠军、2次法国足球甲级联赛冠军、4次欧洲冠军联赛冠军，在国家队获得2022年国际足联世界杯冠军、2014年和2026年国际足联世界杯亚军、1次奥运会足球赛金牌、2次美洲国家杯冠军及1次欧美超级杯冠军。2026年世界杯结束后，梅西宣

In [3]:
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from langchain_tavily import TavilySearch
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
import os
load_dotenv()

# 自定义方法
search_tool = TavilySearch()

# 创建model
model = init_chat_model(
    model="qwen3.5-plus",
    model_provider="openai",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL")
)

# 创建agent
agent = create_agent(
    model,
    tools=[search_tool],
    system_prompt="你是一个智能助手，使用工具来回答用户问题"
)

response = agent.invoke(
    {"messages":[HumanMessage("今天北京天气怎么样？")]}
)

for message in response["messages"]:
    message.pretty_print()


KeyboardInterrupt: 

In [9]:
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from langchain_tavily import TavilySearch
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from pydantic import BaseModel,Field
import os
load_dotenv()

# 自定义方法
search_tool = TavilySearch(
    max_results=3,
    timeout=15,          # 关键：默认 None 会一直卡到超时
    max_retries=2,
)
# 创建model
model = init_chat_model(
    model="qwen3.5-plus",
    model_provider="openai",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL")
)
# Agent所引用的网页信息
class Reference(BaseModel):
    title: str = Field(description="网页标题")
    url: str = Field(description="网页网址")

# Agent的回答内容
class AnswerInfo(BaseModel):
    answer: str = Field(description="最终的答案")
    reference: list[Reference] = Field(description="回答问题的网页信息")
# 创建agent
agent = create_agent(
    model,
    tools=[search_tool],
    system_prompt="你是一个智能助手，使用工具来回答用户问题",
    response_format=AnswerInfo
)

response = agent.invoke(
    {"messages":[HumanMessage("梅西是谁？")]}
)

print(response["structured_response"])


None
